# Safehouse next-month incident forecast

Forecast next-month incident volume from `safehouse_monthly_metrics.csv` using lag features per safehouse.

- **Label**: `incident_count_next_month` (regression)
- **Leakage avoidance**: walk-forward style time split (last 20% months holdout)
- **Output**: `ml-outputs/safehouse-incident-forecast/predictions.csv` (holdout predictions)

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

DATA_DIR = '../lighthouse_csv_v7'
OUT_DIR = '../ml-outputs/safehouse-incident-forecast'
os.makedirs(OUT_DIR, exist_ok=True)

TEST_SIZE = 0.2
RANDOM_STATE = 42

print('Output folder:', OUT_DIR)

Output folder: ../ml-outputs/safehouse-incident-forecast


In [2]:
m = pd.read_csv(os.path.join(DATA_DIR, 'safehouse_monthly_metrics.csv'))

# Dates
m['month_start'] = pd.to_datetime(m['month_start'], errors='coerce', utc=True)

req = {'safehouse_id', 'month_start', 'incident_count'}
missing = req - set(m.columns)
if missing:
    raise ValueError(f"Missing required cols in safehouse_monthly_metrics.csv: {sorted(missing)}")

# Sort
m = m.dropna(subset=['safehouse_id', 'month_start']).copy()
m = m.sort_values(['safehouse_id', 'month_start']).reset_index(drop=True)

# Numeric cols
for c in ['active_residents', 'avg_education_progress', 'avg_health_score', 'process_recording_count', 'home_visitation_count', 'incident_count']:
    if c in m.columns:
        m[c] = pd.to_numeric(m[c], errors='coerce')

# Create lag features within safehouse
lag_cols = ['active_residents', 'avg_education_progress', 'avg_health_score', 'process_recording_count', 'home_visitation_count', 'incident_count']
for c in lag_cols:
    if c in m.columns:
        m[f'{c}_lag1'] = m.groupby('safehouse_id')[c].shift(1)
        m[f'{c}_lag2'] = m.groupby('safehouse_id')[c].shift(2)

# Label: next month's incident count
m['incident_count_next_month'] = m.groupby('safehouse_id')['incident_count'].shift(-1)

# Seasonality
m['month'] = m['month_start'].dt.month
m['year'] = m['month_start'].dt.year

# Drop rows without lag history or label
model_df = m.dropna(subset=['incident_count_next_month', 'incident_count_lag1']).copy()

print('Rows:', len(model_df))
print('Safehouses:', model_df['safehouse_id'].nunique())
model_df[['safehouse_id', 'month_start', 'incident_count_lag1', 'incident_count_next_month']].head()

Rows: 432
Safehouses: 9


,safehouse_id,month_start,incident_count_lag1,incident_count_next_month
1,1,2023-02-01 00:00:00+00:00,0.0,0.0
2,1,2023-03-01 00:00:00+00:00,0.0,1.0
3,1,2023-04-01 00:00:00+00:00,0.0,0.0
4,1,2023-05-01 00:00:00+00:00,1.0,0.0
5,1,2023-06-01 00:00:00+00:00,0.0,0.0


In [3]:
# Time-based split: last TEST_SIZE fraction of months as holdout
cutoff = model_df['month_start'].quantile(1 - TEST_SIZE)
train_df = model_df[model_df['month_start'] <= cutoff].copy()
test_df = model_df[model_df['month_start'] > cutoff].copy()

label_col = 'incident_count_next_month'

feature_cols = [
    'safehouse_id', 'month', 'year',
    'active_residents_lag1', 'active_residents_lag2',
    'avg_education_progress_lag1', 'avg_education_progress_lag2',
    'avg_health_score_lag1', 'avg_health_score_lag2',
    'process_recording_count_lag1', 'process_recording_count_lag2',
    'home_visitation_count_lag1', 'home_visitation_count_lag2',
    'incident_count_lag1', 'incident_count_lag2',
]
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
y_train = train_df[label_col]
X_test = test_df[feature_cols]
y_test = test_df[label_col]

print('Train rows:', len(train_df), 'Test rows:', len(test_df))
print('Feature cols:', feature_cols)

Train rows: 351 Test rows: 81
Feature cols: ['safehouse_id', 'month', 'year', 'active_residents_lag1', 'active_residents_lag2', 'avg_education_progress_lag1', 'avg_education_progress_lag2', 'avg_health_score_lag1', 'avg_health_score_lag2', 'process_recording_count_lag1', 'process_recording_count_lag2', 'home_visitation_count_lag1', 'home_visitation_count_lag2', 'incident_count_lag1', 'incident_count_lag2']


In [4]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
# Treat safehouse_id as categorical for pooling across safehouses
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

models = {
    'Poisson': PoissonRegressor(alpha=0.1, max_iter=2000),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2,
    ),
}

rows = []
fitters = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    pred = np.clip(pred, 0, None)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))

    rows.append({'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2})
    fitters[name] = pipe

results = pd.DataFrame(rows).sort_values(['mae', 'rmse']).reset_index(drop=True)
results

TypeError: got an unexpected keyword argument 'squared'

In [ ]:
best_name = results.loc[0, 'model']
best = fitters[best_name]

pred = np.clip(best.predict(X_test), 0, None)

out = test_df[['safehouse_id', 'month_start', 'incident_count', 'incident_count_lag1']].copy()
out['y_true_next_month'] = y_test.values
out['y_pred_next_month'] = pred
out['abs_error'] = (out['y_true_next_month'] - out['y_pred_next_month']).abs()

out_path = os.path.join(OUT_DIR, 'predictions.csv')
out.to_csv(out_path, index=False)

print('Best model:', best_name)
print('Wrote:', out_path)

out.sort_values('abs_error', ascending=False).head(10)

## Website surfacing + exported CSV schema

### Admin dashboard
- Show next-month incident forecast per safehouse (`y_pred_next_month`) to inform staffing/training.

### Public impact page (public-safe)
- If shown publicly, keep it high-level (aggregate/organization-wide) and avoid identifying locations if that’s sensitive.

### Exported files
- `ml-outputs/safehouse-incident-forecast/predictions.csv`
  - `safehouse_id`, `month_start`, `incident_count` (current), `incident_count_lag1`, `y_true_next_month`, `y_pred_next_month`, `abs_error`